# Chunking Strategies Benchmark — Failure-Mode Analysis Across All Data Types

This notebook evaluates **three chunking strategies** across **seven distinct
data types** found in educational documents (curriculum PDFs, student exam
submissions, code, maths, tables, diagrams). The immediate goal is to measure
whether chunking preserves meaningful structure. Retrieval and production
embedding-model selection are deliberately deferred to a later stage.

**Why this matters.** Retrieval can only return what chunking preserved. If a
chunk boundary severs a table header from its rows, splits a code block
mid-function, or isolates a formula from its definition, later embedding and
search layers cannot recover that lost context.

**How to read this notebook.** Each step does one thing, prints its output, and
explains what happened before the next step starts. This follows the
[NOTEBOOK_STANDARD.md](../../02-experiments/NOTEBOOK_STANDARD.md) convention:
one action per cell, so a reader can find the exact line that broke.

**Embedding boundary.** The fixed and Hybrid strategies in this notebook do not
call an embedding model. The semantic strategy uses only a deterministic,
offline hash-vector stub so its grouping code can run without credentials; that
stub is not a quality measurement. `max_tokens` in the Hybrid experiments is
Docling's chunking budget. The notebook reports final token sizes for later
model selection, but it does not truncate text or claim to test retrieval.

## What this notebook builds

| Name | What it does | Example |
| --- | --- | --- |
| `FIXTURE_PATH` | Path to the 7-data-type benchmark document | `fixture_all_types.md` |
| `check_table_header_retention(chunks)` | Checks whether chunks containing table rows also contain the column header row | Returns `{"total_table_chunks": 3, "with_header": 2, "retention_rate": 0.67}` |
| `check_code_block_integrity(chunks)` | Checks whether code fence markers are balanced within each chunk | Returns `{"total_code_chunks": 1, "intact": 1, "broken": 0}` |
| `check_math_integrity(chunks)` | Checks whether `$` delimiters are balanced within each chunk | Returns `{"total_math_chunks": 5, "balanced": 4, "unbalanced": 1}` |
| `check_heading_context(chunks)` | Checks whether chunks carry heading/section context | Returns `{"with_heading": 8, "without_heading": 2}` |
| `check_qa_integrity(chunks)` | Checks whether question prompts and student answers stay together | Returns `{"question_found": True, "answer_found": True, "same_chunk": True}` |
| `check_diagram_integrity(chunks)` | Checks whether diagram/chart content stays with its caption | Returns `{"caption_found": True, "visual_found": True, "same_chunk": False}` |
| `summarise_chunks(chunks)` | Prints count, min/max/mean word counts | `"12 chunks, 45–410 words, mean 210"` |
| `run_fixed_splitter(text, ...)` | Fixed-size chunker with configurable size, overlap, and separators | `run_fixed_splitter(text, chunk_size=400)` |
| `run_markdown_splitter(text, ...)` | LangChain's `MarkdownTextSplitter` — a Markdown-aware fixed splitter | `run_markdown_splitter(text, chunk_size=400)` |
| `run_semantic_chunker(text, ...)` | Sentence-split → embed → cosine-group → merge | `run_semantic_chunker(text, threshold=0.7)` |
| `run_hybrid_chunker(md_path, ...)` | Docling `HybridChunker` with full feature set | `run_hybrid_chunker(path, max_tokens=512)` |

## The seven data types in the fixture

| # | Data type | Where it appears in real documents | What breaks if chunked badly |
| --- | --- | --- | --- |
| 1 | **Continuous prose** | Curriculum rationale, forewords, syllabi | Mid-sentence cuts disrupt reading flow |
| 2 | **Hierarchical headings** | Grade > Strand > Sub-strand > Indicator | Chunks lose parent context ("which grade is this?") |
| 3 | **Dense tables** | Content standards, indicators, exemplars | Column headers stranded in earlier chunk; rows become meaningless numbers |
| 4 | **Code blocks** | Grading algorithms, student programming assignments | Indentation, scope, and syntax broken mid-function |
| 5 | **Math expressions** | Exam answers, worked examples, LaTeX formulas | Unclosed `$` delimiters; fractions split between numerator and denominator |
| 6 | **Student Q&A** | Transcribed exam papers (question + multi-step answer) | Question prompt separated from student's working |
| 7 | **Diagram / chart captions** | Bar charts, coordinate grids, ASCII visuals | Caption detached from the visual it describes |

## Step 1 — Set up the environment

Put the repo root on `sys.path` so `import nbio` works no matter where
Jupyter started the kernel, then run `nbio.bootstrap()`. This is the same
pattern every notebook in this repo uses.

In [1]:
import sys
from pathlib import Path

# Walk up from this notebook's directory to find the repo root (where
# nbio.py lives). Jupyter sets cwd to the notebook's own folder.
_root = Path.cwd().resolve()
for _ in range(6):
    if (_root / "nbio.py").is_file():
        break
    _root = _root.parent
else:
    raise RuntimeError("could not locate nbio.py above the current directory")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import nbio
nbio.bootstrap()
print(f"Repo root: {_root}")

Repo root: C:\Users\saima\Documents\anacodic-agentic-cookbook


## Step 2 — Load the 7-type fixture document

The fixture `fixture_all_types.md` lives next to this notebook. It contains
all seven data types in one file so every chunker is tested against
identical input. Let's load it and look at basic statistics.

In [2]:
FIXTURE_PATH = Path("fixture_all_types.md")
if not FIXTURE_PATH.exists():
    # When run from a different cwd, try relative to this notebook's file
    FIXTURE_PATH = Path(__file__).parent / "fixture_all_types.md" if "__file__" in dir() else FIXTURE_PATH

fixture_text = FIXTURE_PATH.read_text(encoding="utf-8")

print(f"Fixture loaded: {FIXTURE_PATH.name}")
print(f"  Characters : {len(fixture_text):,}")
print(f"  Words      : {len(fixture_text.split()):,}")
print(f"  Lines      : {len(fixture_text.splitlines()):,}")
print()
print("--- First 500 characters ---")
print(fixture_text[:500])

Fixture loaded: fixture_all_types.md
  Characters : 9,869
  Words      : 1,523
  Lines      : 239

--- First 500 characters ---
# Ghana Primary Mathematics Curriculum — Upper Primary (B4–B6)

This document is a benchmark fixture for chunking strategy evaluation. It
contains seven distinct data types, each clearly marked, so automated checks
can verify whether a chunker kept or broke each one.

## Rationale and Philosophy

The Ghana primary mathematics curriculum emphasises the development of
critical thinking and problem-solving abilities from the earliest years of
formal education. Learners are expected to build concept


### Quick content audit — verify all 7 data types are present

Before running any chunker, let's confirm the fixture actually contains all
seven data types. This is a sanity check so we don't accidentally benchmark
against incomplete input.

In [3]:
import re

# Markers that identify each data type in the fixture
DATA_TYPE_MARKERS = {
    "1. Prose":        "critical thinking and problem-solving",
    "2. Headings":     "### Sub-strand",
    "3. Table":        "| Content Standard |",
    "4. Code":         "def grade_arithmetic",
    "5. Math":         "$\\frac{3}{4}",
    "6. Student Q&A":  "**Question 3 (5 marks):",
    "7. Diagram":      "Favourite Fruits Survey",
}

print("Data type audit:")
all_present = True
for name, marker in DATA_TYPE_MARKERS.items():
    found = marker in fixture_text
    status = "✓ FOUND" if found else "✗ MISSING"
    print(f"  {name:20s} {status}")
    if not found:
        all_present = False

assert all_present, "Not all 7 data types found in fixture — check fixture_all_types.md"
print("\nAll 7 data types confirmed present.")

Data type audit:
  1. Prose             ✓ FOUND
  2. Headings          ✓ FOUND
  3. Table             ✓ FOUND
  4. Code              ✓ FOUND
  5. Math              ✓ FOUND
  6. Student Q&A       ✓ FOUND
  7. Diagram           ✓ FOUND

All 7 data types confirmed present.


## Step 3 — Build the diagnostic integrity checkers

These functions inspect a list of chunk strings and report whether each data
type survived chunking intact. They are the "tests" of this benchmark —
instead of eyeballing chunks, we programmatically detect the specific
failure modes that matter.

In [4]:
def check_table_header_retention(chunks: list[str]) -> dict:
    """Check whether chunks containing table data rows also contain the
    column header row.  A table row is any line matching '| ... | ... |'.
    The header row contains 'Content Standard'."""
    header_marker = "Content Standard"
    # A data row has pipes and at least one cell with a B4/B5/B6 indicator
    data_pattern = re.compile(r"\|\s*B[456]")

    table_chunks = []
    for c in chunks:
        if data_pattern.search(c):
            table_chunks.append(c)

    with_header = sum(1 for c in table_chunks if header_marker in c)
    total = len(table_chunks)
    return {
        "total_table_chunks": total,
        "with_header": with_header,
        "without_header": total - with_header,
        "retention_rate": with_header / total if total else 1.0,
    }

### Code block integrity checker

Checks whether code fence markers (` ``` `) are balanced within each chunk.
An odd count means the chunk sliced through the middle of a fenced code
block.

In [5]:
def check_code_block_integrity(chunks: list[str]) -> dict:
    """Check whether code fence markers (```) are balanced in each chunk."""
    code_chunks = [c for c in chunks if "```" in c]
    intact = 0
    broken = 0
    for c in code_chunks:
        fence_count = c.count("```")
        if fence_count % 2 == 0:
            intact += 1
        else:
            broken += 1
    return {
        "total_code_chunks": len(code_chunks),
        "intact": intact,
        "broken": broken,
    }

### Math delimiter integrity checker

Checks whether `$` delimiters are balanced within each chunk. LaTeX math
uses `$...$` for inline and `$$...$$` for display. An odd count of `$`
markers means the chunker sliced between an opening and closing delimiter.

In [6]:
def check_math_integrity(chunks: list[str]) -> dict:
    """Check whether $ delimiters are balanced in each chunk containing math."""
    math_chunks = [c for c in chunks if "$" in c]
    balanced = 0
    unbalanced = 0
    for c in math_chunks:
        # Count standalone $ signs (not escaped \$)
        dollar_count = len(re.findall(r"(?<!\\)\$", c))
        if dollar_count % 2 == 0:
            balanced += 1
        else:
            unbalanced += 1
    return {
        "total_math_chunks": len(math_chunks),
        "balanced": balanced,
        "unbalanced": unbalanced,
    }

### Q&A integrity checker

Checks whether the exam question prompt and the student's answer are in the
same chunk. If they get separated, a retrieval system cannot connect the
student's working to the question it answers.

In [7]:
def check_qa_integrity(chunks: list[str]) -> dict:
    """Check whether Q3's question and student answer land in the same chunk."""
    question_marker = "A farmer has 2,450 oranges"
    answer_marker = "70 full crates"

    q_chunks = [i for i, c in enumerate(chunks) if question_marker in c]
    a_chunks = [i for i, c in enumerate(chunks) if answer_marker in c]

    same_chunk = bool(set(q_chunks) & set(a_chunks))
    return {
        "question_found": len(q_chunks) > 0,
        "answer_found": len(a_chunks) > 0,
        "same_chunk": same_chunk,
        "question_in_chunks": q_chunks,
        "answer_in_chunks": a_chunks,
    }

### Diagram/caption integrity checker

Checks whether the ASCII chart and its caption stay together.

In [8]:
def check_diagram_integrity(chunks: list[str]) -> dict:
    """Check whether Figure 1's ASCII bar chart and its caption are in the
    same chunk."""
    visual_marker = "Mango       |    12"
    caption_marker = "modal fruit is mango"

    v_chunks = [i for i, c in enumerate(chunks) if visual_marker in c]
    c_chunks = [i for i, c in enumerate(chunks) if caption_marker in c]

    same_chunk = bool(set(v_chunks) & set(c_chunks))
    return {
        "visual_found": len(v_chunks) > 0,
        "caption_found": len(c_chunks) > 0,
        "same_chunk": same_chunk,
    }

### Chunk statistics helper

In [9]:
def summarise_chunks(chunks: list[str], label: str = "") -> dict:
    """Print and return chunk count, min/max/mean word counts."""
    word_counts = [len(c.split()) for c in chunks]
    stats = {
        "count": len(chunks),
        "min_words": min(word_counts) if word_counts else 0,
        "max_words": max(word_counts) if word_counts else 0,
        "mean_words": round(sum(word_counts) / len(word_counts), 1) if word_counts else 0,
    }
    prefix = f"[{label}] " if label else ""
    print(f"{prefix}{stats['count']} chunks, "
          f"{stats['min_words']}–{stats['max_words']} words, "
          f"mean {stats['mean_words']}")
    return stats

### Run all diagnostic checks on a chunk list

In [10]:
def run_all_checks(chunks: list[str], label: str = "") -> dict:
    """Run every integrity check and return a combined report."""
    stats   = summarise_chunks(chunks, label)
    table   = check_table_header_retention(chunks)
    code    = check_code_block_integrity(chunks)
    math    = check_math_integrity(chunks)
    qa      = check_qa_integrity(chunks)
    diagram = check_diagram_integrity(chunks)

    print(f"  Table header retention : {table['retention_rate']:.0%} "
          f"({table['with_header']}/{table['total_table_chunks']} table chunks keep header)")
    print(f"  Code block integrity   : {code['intact']}/{code['total_code_chunks']} intact")
    print(f"  Math delimiter balance : {math['balanced']}/{math['total_math_chunks']} balanced")
    print(f"  Q&A same chunk         : {qa['same_chunk']}")
    print(f"  Diagram+caption same   : {diagram['same_chunk']}")

    return {
        "label": label,
        **stats,
        "table_retention": table["retention_rate"],
        "code_intact": code["intact"] == code["total_code_chunks"],
        "math_balanced": math["balanced"] == math["total_math_chunks"],
        "qa_together": qa["same_chunk"],
        "diagram_together": diagram["same_chunk"],
    }

---

## Step 4 — Strategy 1: Fixed / Recursive Character Splitting

This is the baseline chunker from `01-modules/02-chunk/01-character-splitting.ipynb`.
It uses LangChain's `RecursiveCharacterTextSplitter`, which packs text into
fixed-size chunks using a preference-ordered list of separators. It has **no
awareness of document structure** — it cannot see tables, headings, code
fences, or math delimiters.

### Features we will test:

1. **Custom separators** — `MEDICAL_SEPARATORS` (section headers → paragraph → sentence → word)
2. **Markdown-aware separators** — `MarkdownTextSplitter` (splits on `#`, `##`, `###`, code fences)
3. **`chunk_size`** — word budget: 200, 400, 800
4. **`chunk_overlap`** — 0 words (no overlap) vs 50 words
5. **`length_function`** — word count (`len(t.split())`)
6. **`keep_separator`** — whether separators are retained at chunk edges

The point: *no matter how cleverly you configure separators, a flat-text
splitter has no representation of "table", "row", or "header" — so it
cannot protect a boundary it cannot see.*

In [11]:
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    MarkdownTextSplitter,
    Language,
)

# --- Separator lists ---

# 1. Default separators (paragraph → line → sentence → word)
DEFAULT_SEPARATORS = ["\n\n", "\n", ". ", " ", ""]

# 2. Heading-aware separators (tries markdown headings first)
HEADING_SEPARATORS = [
    "\n# ", "\n## ", "\n### ", "\n#### ",  # Markdown headings
    "\n\n", "\n",                             # Paragraphs, lines
    ". ", " ", "",                             # Sentences, words
]

print("Separator lists defined:")
print(f"  DEFAULT_SEPARATORS  : {len(DEFAULT_SEPARATORS)} separators")
print(f"  HEADING_SEPARATORS  : {len(HEADING_SEPARATORS)} separators")
print(f"  MarkdownTextSplitter: LangChain built-in (splits on Markdown syntax)")

c:\Users\saima\Documents\anacodic-agentic-cookbook\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Separator lists defined:
  DEFAULT_SEPARATORS  : 5 separators
  HEADING_SEPARATORS  : 9 separators
  MarkdownTextSplitter: LangChain built-in (splits on Markdown syntax)


### Define the fixed splitter runner

This function wraps `RecursiveCharacterTextSplitter` so we can sweep
parameters cleanly.

In [12]:
def run_fixed_splitter(
    text: str,
    chunk_size: int = 400,
    chunk_overlap: int = 50,
    separators: list[str] | None = None,
    keep_separator: bool | str = True,
) -> list[str]:
    """Run RecursiveCharacterTextSplitter with the given parameters.

    Parameters
    ----------
    text : str
        The full document text to chunk.
    chunk_size : int
        Target chunk size in words.
    chunk_overlap : int
        Overlap between consecutive chunks, in words.
    separators : list[str] or None
        Separator preference order. Defaults to DEFAULT_SEPARATORS.
    keep_separator : bool or str
        Whether to keep separators — True, False, "start", or "end".

    Returns
    -------
    list[str]
        The resulting chunks.
    """
    if separators is None:
        separators = DEFAULT_SEPARATORS

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=separators,
        length_function=lambda t: len(t.split()),
        keep_separator=keep_separator,
    )
    return splitter.split_text(text)

### Define the Markdown-aware splitter runner

`MarkdownTextSplitter` is LangChain's built-in splitter that knows about
Markdown syntax — it splits on `#` headings, code fences (` ``` `), and
horizontal rules (`---`) before falling back to paragraph and sentence
boundaries. This is a feature of the Fixed strategy family that we must
test to be fair — we cannot say "fixed splitting breaks on headings" if we
didn't try the Markdown-aware variant.

In [13]:
def run_markdown_splitter(
    text: str,
    chunk_size: int = 400,
    chunk_overlap: int = 50,
) -> list[str]:
    """Run LangChain's MarkdownTextSplitter.

    This splitter uses Markdown-specific separators:
    code fences, headings (# ## ### ...), horizontal rules,
    then falls back to paragraphs, sentences, words.
    """
    splitter = MarkdownTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=lambda t: len(t.split()),
    )
    return splitter.split_text(text)

### Run the fixed splitter across parameter configurations

We sweep across chunk sizes (200, 400, 800 words), overlaps (0, 50 words),
and separator strategies (default, heading-aware, Markdown-aware).

In [14]:
fixed_results = []

# Configuration sweep
configs = [
    # (label, chunk_size, overlap, separators, use_markdown_splitter)
    ("default-200w-0ov",     200, 0,  DEFAULT_SEPARATORS, False),
    ("default-400w-0ov",     400, 0,  DEFAULT_SEPARATORS, False),
    ("default-400w-50ov",    400, 50, DEFAULT_SEPARATORS, False),
    ("default-800w-0ov",     800, 0,  DEFAULT_SEPARATORS, False),
    ("default-800w-50ov",    800, 50, DEFAULT_SEPARATORS, False),
    ("heading-400w-50ov",    400, 50, HEADING_SEPARATORS, False),
    ("heading-800w-50ov",    800, 50, HEADING_SEPARATORS, False),
    ("markdown-400w-50ov",   400, 50, None, True),
    ("markdown-800w-50ov",   800, 50, None, True),
]

print("=" * 72)
print("STRATEGY 1: Fixed / Recursive Splitting")
print("=" * 72)

for label, size, overlap, seps, use_md in configs:
    print(f"\n--- {label} ---")
    if use_md:
        chunks = run_markdown_splitter(fixture_text, chunk_size=size, chunk_overlap=overlap)
    else:
        chunks = run_fixed_splitter(fixture_text, chunk_size=size, chunk_overlap=overlap, separators=seps)

    report = run_all_checks(chunks, label)
    fixed_results.append(report)

print("\n" + "=" * 72)

STRATEGY 1: Fixed / Recursive Splitting

--- default-200w-0ov ---
[default-200w-0ov] 10 chunks, 58–195 words, mean 152.3
  Table header retention : 50% (1/2 table chunks keep header)
  Code block integrity   : 2/2 intact
  Math delimiter balance : 2/2 balanced
  Q&A same chunk         : True
  Diagram+caption same   : True

--- default-400w-0ov ---
[default-400w-0ov] 4 chunks, 352–399 words, mean 380.8
  Table header retention : 100% (1/1 table chunks keep header)
  Code block integrity   : 2/2 intact
  Math delimiter balance : 2/2 balanced
  Q&A same chunk         : False
  Diagram+caption same   : True

--- default-400w-50ov ---
[default-400w-50ov] 5 chunks, 97–392 words, mean 329.6
  Table header retention : 100% (1/1 table chunks keep header)
  Code block integrity   : 3/3 intact
  Math delimiter balance : 2/2 balanced
  Q&A same chunk         : True
  Diagram+caption same   : True

--- default-800w-0ov ---
[default-800w-0ov] 2 chunks, 734–789 words, mean 761.5
  Table header reten

### Inspect a specific failure: the table cut

Let's zoom into the `default-400w-0ov` configuration and look at exactly
where the table got split. This is the core failure mode: the column header
row (`| Content Standard | Indicator | ...`) ends up in an earlier chunk,
and a later chunk has raw data rows with no header — making those rows
uninterpretable.

In [15]:
# Rerun the 400w config to inspect chunks
inspect_chunks = run_fixed_splitter(fixture_text, chunk_size=400, chunk_overlap=0)

data_row_pattern = re.compile(r"\|\s*B[456]")
header_marker = "Content Standard"

print(f"Total chunks: {len(inspect_chunks)}\n")
for i, c in enumerate(inspect_chunks):
    has_table_data = bool(data_row_pattern.search(c))
    has_header = header_marker in c
    has_code = "```" in c
    has_math = "$" in c

    flags = []
    if has_table_data: flags.append("TABLE-DATA")
    if has_header:     flags.append("TABLE-HEADER")
    if has_code:       flags.append("CODE")
    if has_math:       flags.append("MATH")

    flag_str = ", ".join(flags) if flags else "prose-only"
    print(f"chunk {i:2d}  ({len(c.split()):3d} words)  [{flag_str}]")
    # Show first 120 chars for chunks with table data but no header
    if has_table_data and not has_header:
        print(f"    ⚠️  TABLE DATA WITHOUT HEADER — first 120 chars:")
        print(f"    {c[:120]}")

Total chunks: 4

chunk  0  (386 words)  [TABLE-HEADER]
chunk  1  (386 words)  [TABLE-DATA, TABLE-HEADER]
chunk  2  (399 words)  [CODE, MATH]
chunk  3  (352 words)  [CODE, MATH]


---

## Step 5 — Strategy 2: Semantic / Adaptive Chunking

This strategy splits text into sentences, embeds each sentence, measures
cosine similarity between consecutive sentences, and creates a chunk boundary
wherever similarity drops below a threshold τ. The idea: chunk boundaries
should fall where *meaning* changes, not at an arbitrary word count.

### Features we will test:

1. **Sentence splitting** — regex `(?<=[.!?])\s+` with protection for decimals
2. **Similarity threshold τ** — sweep from 0.50 to 0.90
3. **`min_chunk_words`** — floor to prevent single-sentence chunks
4. **`max_sentences`** — ceiling to prevent runaway embedding
5. **Embedding function** — deterministic hash stub (offline, no API)

### Known weaknesses of this approach:

- **Tables**: Pipe-delimited rows (`| A | B |`) have no `.!?` punctuation,
  so the sentence splitter either treats the entire table as one "sentence"
  or breaks on decimal points inside numbers.
- **Code blocks**: Code lines end with `:`, `)`, or nothing — not `.!?`.
- **Threshold sensitivity**: Too high → over-fragments; too low → merges
  unrelated content.

In [16]:
import hashlib
import math


def _get_sentences(text: str, max_sentences: int = 300) -> list[str]:
    """Split text into sentences on .!? boundaries.

    This regex approach is simple and has known limitations:
    - It will split on decimal points in numbers like '3.14'
    - It cannot handle abbreviations like 'e.g.' or 'i.e.'
    - Table rows and code lines typically don't end with .!?
    """
    if not text or not text.strip():
        return []
    parts = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s.strip() for s in parts if s.strip()][:max_sentences]


def _cosine_similarity(a: list[float], b: list[float]) -> float:
    """Cosine similarity between two vectors."""
    dot = sum(x * y for x, y in zip(a, b))
    na = math.sqrt(sum(x * x for x in a))
    nb = math.sqrt(sum(x * x for x in b))
    if na == 0 or nb == 0:
        return 0.0
    return dot / (na * nb)


def hash_embed_stub(texts: list[str], dim: int = 16) -> list[list[float]]:
    """Deterministic, offline, NOT semantically meaningful.

    Exists only so the adaptive chunker has something to call without a
    real API key. Swap for a real embedding call (stage 03) to get real
    similarity grouping.
    """
    out = []
    for t in texts:
        h = hashlib.sha256(t.encode("utf-8")).digest()
        vec = [b / 255.0 for b in h[:dim]]
        out.append(vec)
    return out


print("Sentence splitting, cosine similarity, and hash embedding stub defined.")

Sentence splitting, cosine similarity, and hash embedding stub defined.


### Define the semantic/adaptive chunker runner

In [17]:
def _adaptive_chunk_groups(
    sentences: list[str],
    embeddings: list[list[float]],
    min_similarity: float = 0.6,
) -> list[list[int]]:
    """Group adjacent sentence indices whose cosine similarity clears the
    threshold. A chunk boundary falls wherever two consecutive sentences
    stop being 'about the same thing'."""
    if not sentences or not embeddings:
        return []
    if len(sentences) != len(embeddings):
        return [[i] for i in range(len(sentences))]

    groups, current = [], [0]
    for i in range(1, len(sentences)):
        sim = _cosine_similarity(embeddings[i - 1], embeddings[i])
        if sim >= min_similarity:
            current.append(i)
        else:
            groups.append(current)
            current = [i]
    groups.append(current)
    return groups


def run_semantic_chunker(
    text: str,
    threshold: float = 0.7,
    min_chunk_words: int = 30,
    embed_fn=None,
) -> list[str]:
    """Adaptive chunking: sentence-split → embed → cosine-group → merge.

    Parameters
    ----------
    text : str
        The full document text.
    threshold : float
        Minimum cosine similarity to keep sentences in the same group.
        Higher = more splits (over-fragmentation risk).
        Lower = fewer splits (under-splitting risk).
    min_chunk_words : int
        Minimum word count for a chunk to be kept.
    embed_fn : callable or None
        Embedding function. Defaults to hash_embed_stub.
    """
    if embed_fn is None:
        embed_fn = hash_embed_stub

    sentences = _get_sentences(text)
    if not sentences:
        return [text] if text.strip() else []
    if len(sentences) == 1:
        return [text] if len(text.split()) >= min_chunk_words else []

    embeddings = embed_fn(sentences)
    groups = _adaptive_chunk_groups(sentences, embeddings, min_similarity=threshold)

    chunks = []
    for g in groups:
        chunk = " ".join(sentences[i] for i in g)
        if len(chunk.split()) >= min_chunk_words:
            chunks.append(chunk)

    return chunks if chunks else [text]


print("run_semantic_chunker defined.")

run_semantic_chunker defined.


### Run the semantic chunker across threshold configurations

In [18]:
semantic_results = []

thresholds = [0.50, 0.60, 0.70, 0.80, 0.90]

print("=" * 72)
print("STRATEGY 2: Semantic / Adaptive Chunking")
print("=" * 72)

for tau in thresholds:
    label = f"semantic-τ={tau:.2f}"
    print(f"\n--- {label} ---")
    chunks = run_semantic_chunker(fixture_text, threshold=tau)
    report = run_all_checks(chunks, label)
    semantic_results.append(report)

print("\n" + "=" * 72)

STRATEGY 2: Semantic / Adaptive Chunking

--- semantic-τ=0.50 ---
[semantic-τ=0.50] 1 chunks, 1523–1523 words, mean 1523.0
  Table header retention : 100% (1/1 table chunks keep header)
  Code block integrity   : 1/1 intact
  Math delimiter balance : 1/1 balanced
  Q&A same chunk         : True
  Diagram+caption same   : True

--- semantic-τ=0.60 ---
[semantic-τ=0.60] 3 chunks, 41–1076 words, mean 507.7
  Table header retention : 100% (1/1 table chunks keep header)
  Code block integrity   : 2/2 intact
  Math delimiter balance : 2/2 balanced
  Q&A same chunk         : True
  Diagram+caption same   : True

--- semantic-τ=0.70 ---
[semantic-τ=0.70] 10 chunks, 31–381 words, mean 135.3
  Table header retention : 100% (1/1 table chunks keep header)
  Code block integrity   : 1/3 intact
  Math delimiter balance : 3/3 balanced
  Q&A same chunk         : False
  Diagram+caption same   : True

--- semantic-τ=0.80 ---
[semantic-τ=0.80] 12 chunks, 35–236 words, mean 89.2
  Table header retention 

### Inspect: how does the sentence splitter handle tables and code?

Let's look at what `_get_sentences` does to our fixture's table and code
sections specifically. This reveals *why* the semantic chunker struggles
with structured content.

In [19]:
# Extract just the table section
table_start = fixture_text.find("| Content Standard |")
table_end = fixture_text.find("## Sample Grading Algorithm")
table_section = fixture_text[table_start:table_end].strip()

table_sentences = _get_sentences(table_section)
print(f"Table section split into {len(table_sentences)} 'sentences':")
for i, s in enumerate(table_sentences[:8]):
    print(f"  [{i}] {s[:100]}{'...' if len(s) > 100 else ''}")

print()

# Extract just the code section
code_start = fixture_text.find("```python")
code_end = fixture_text.find("```", code_start + 10) + 3
code_section = fixture_text[code_start:code_end]

code_sentences = _get_sentences(code_section)
print(f"Code block split into {len(code_sentences)} 'sentences':")
for i, s in enumerate(code_sentences[:8]):
    print(f"  [{i}] {s[:100]}{'...' if len(s) > 100 else ''}")

Table section split into 2 'sentences':
  [0] | Content Standard | Indicator | Exemplar | Level | ICT Integration |
| --- | --- | --- | --- | --- ...
  [1] → GHS 48 | B6 | Spreadsheet proportional fill |
| B6.1.5.1 Percentages | B6.1.5.1.1 Express fraction...

Code block split into 5 'sentences':
  [0] ```python
def grade_arithmetic(student_answer: str, correct_answer: float,
                     tole...
  [1] Parameters
    ----------
    student_answer : str
        The student's raw text answer (may includ...
  [2] correct_answer : float
        The numerically correct value.
  [3] tolerance : float
        Acceptable relative error (default 1%).
  [4] Returns
    -------
    dict
        {"score": 0 or 1, "feedback": str}
    """
    # Strip non-nume...


---

## Step 6 — Strategy 3: Docling Hybrid Chunker

This is the structure-aware chunker from
`01-modules/02-chunk/02-token-aware-chunking.ipynb`. It walks Docling's
document layout tree rather than a flat string — it knows what a `TableItem`,
`SectionHeaderItem`, and `TextItem` are, so chunk boundaries respect
structural units.

### Features we will test:

1. **`max_tokens`** — Docling's token budget: 256, 512, 1024
2. **`repeat_table_header`** — `True` vs `False` when a table must be split
3. **`merge_peers`** — `True` vs `False` for adjacent same-type items
4. **`always_emit_headings`** — `True` vs `False` for heading attachment
5. **`contextualize(chunk)`** — prepend section context to `embed_text`
6. **`chunk_type` tagging** — automatic `"table"` / `"figure"` / `"prose"`
7. **table export inspection** — compare the complete Markdown export with the
   actual piece emitted by HybridChunker
8. **token overlap reconstruction** — `_tail_by_tokens()` for optional overlap

The runner keeps `chunk.text` as the canonical emitted piece. The complete
`export_to_markdown()` result is retained as diagnostic metadata only; replacing
every split piece with the complete table would duplicate content and invalidate
the token-budget comparison.

### Why this chunker needs a `DoclingDocument`, not a string:

We convert our Markdown fixture into a `DoclingDocument` using Docling's
built-in markdown backend. This is a local operation (no model download, no
network) that parses `#` headings into a heading tree and `| pipe |` tables
into real `TableItem` objects.

A transformer tokenizer warning may appear when Docling inspects a large
structural item whose token count exceeds the tokenizer model's own 512-token
limit. That warning comes from Docling's internal tokenization path; this
notebook does not send the text to an embedding model or silently truncate it.
The final record diagnostics show whether the emitted/enriched chunk exceeds
the configured Hybrid budget.

In [22]:
import os
import tempfile
from typing import Any

from docling.document_converter import DocumentConverter
from docling.chunking import HybridChunker


def make_docling_doc(md_text: str):
    """Convert a Markdown string into a Docling DoclingDocument.

    Uses Docling's markdown backend - local, no model download, no
    network. The markdown parser turns:
    - # headings -> SectionHeaderItem nodes
    - | pipe | tables -> TableItem objects
    - ``` code fences -> CodeItem objects
    - paragraphs -> TextItem objects
    """
    with tempfile.NamedTemporaryFile(
        mode="w", suffix=".md", encoding="utf-8", delete=False
    ) as handle:
        handle.write(md_text)
        tmp_path = Path(handle.name)

    try:
        return DocumentConverter().convert(str(tmp_path)).document
    finally:
        try:
            os.unlink(tmp_path)
        except PermissionError:
            # Docling can still hold the path briefly on Windows.
            pass


# Convert the fixture
print("Converting fixture to DoclingDocument (local markdown parsing)...")
docling_doc = make_docling_doc(fixture_text)

# Inspect what Docling found
n_tables = len(docling_doc.tables) if hasattr(docling_doc, 'tables') else 0
n_pictures = len(docling_doc.pictures) if hasattr(docling_doc, 'pictures') else 0
print(f"DoclingDocument created:")
print(f"  Tables   : {n_tables}")
print(f"  Pictures : {n_pictures}")
print(f"  Markdown export preview (first 300 chars):")
print(docling_doc.export_to_markdown()[:300])

Converting fixture to DoclingDocument (local markdown parsing)...
DoclingDocument created:
  Tables   : 1
  Pictures : 0
  Markdown export preview (first 300 chars):
# Ghana Primary Mathematics Curriculum — Upper Primary (B4–B6)

This document is a benchmark fixture for chunking strategy evaluation. It contains seven distinct data types, each clearly marked, so automated checks can verify whether a chunker kept or broke each one.

## Rationale and Philosophy

Th


### Define helper functions for the Hybrid Chunker

These are ported from `02-token-aware-chunking.ipynb` with full
documentation. They handle the version-drift shim (resolving stub
references to real objects), chunk-type tagging, table extraction in
multiple formats, and token-based overlap reconstruction.

In [23]:
MIN_CHUNK_WORDS = 25


def _item_label(item: Any) -> str:
    """Read an item's label off either a stub or a fully-typed object."""
    label = getattr(item, "label", "")
    if hasattr(label, "value"):
        label = label.value
    return str(label).lower()


def _resolve_doc_item(item: Any, doc: Any) -> Any:
    """Resolve a doc_items stub back to the fully-typed TableItem/PictureItem.

    In docling-core 2.86+, chunk.meta.doc_items returns lightweight stubs
    that carry .label and .self_ref but NOT the subclass methods like
    export_to_markdown() or .data.  This function parses self_ref
    (e.g. '#/tables/0') and looks up the real object in doc.tables[i].
    """
    ref = getattr(item, "self_ref", "") or ""
    try:
        if ref.startswith("#/tables/"):
            return doc.tables[int(ref.rsplit("/", 1)[-1])]
        if ref.startswith("#/pictures/"):
            return doc.pictures[int(ref.rsplit("/", 1)[-1])]
    except (IndexError, ValueError):
        pass
    return item


def _try_table_markdown(doc_items: list[Any], doc: Any) -> str:
    """Extract a table's clean Markdown export via Docling's native method."""
    for item in doc_items:
        resolved = _resolve_doc_item(item, doc)
        if _item_label(resolved) == "table" and hasattr(resolved, "export_to_markdown"):
            try:
                return resolved.export_to_markdown(doc=doc)
            except Exception:
                pass
    return ""


def _try_table_structured(doc_items: list[Any], doc: Any) -> "dict[str, Any] | None":
    """Extract a table as a structured DataFrame-shaped dictionary."""
    for item in doc_items:
        resolved = _resolve_doc_item(item, doc)
        if _item_label(resolved) == "table" and hasattr(resolved, "export_to_dataframe"):
            try:
                df = resolved.export_to_dataframe(doc=doc)
                return {
                    "headers": df.columns.tolist() if hasattr(df, "columns") else [],
                    "rows": df.to_dict("records") if hasattr(df, "to_dict") else [],
                    "num_rows": len(df),
                    "num_cols": len(df.columns) if hasattr(df, "columns") else 0,
                }
            except Exception:
                pass
    return None


def _extract_table_cells(doc_items: list[Any], doc: Any) -> "list[dict[str, Any]] | None":
    """Extract cell-level metadata with spans and header flags."""
    for item in doc_items:
        resolved = _resolve_doc_item(item, doc)
        if _item_label(resolved) == "table":
            data = getattr(resolved, "data", None)
            if data and hasattr(data, "table_cells"):
                cells = []
                for cell in data.table_cells:
                    cells.append({
                        "text": getattr(cell, "text", ""),
                        "row": getattr(cell, "start_row_offset_idx", 0),
                        "col": getattr(cell, "start_col_offset_idx", 0),
                        "row_span": getattr(cell, "row_span", 1),
                        "col_span": getattr(cell, "col_span", 1),
                        "is_header": bool(
                            getattr(cell, "column_header", False)
                            or getattr(cell, "row_header", False)
                        ),
                    })
                return cells
    return None


def get_chunk_type(chunk_obj: Any, doc: Any) -> str:
    """Tag a chunk as 'table', 'figure', or 'prose'."""
    doc_items = list(getattr(getattr(chunk_obj, "meta", None), "doc_items", None) or [])
    labels = set()
    for item in doc_items:
        resolved = _resolve_doc_item(item, doc)
        labels.add(_item_label(resolved))
    if "table" in labels:
        return "table"
    if labels & {"picture", "figure"}:
        return "figure"
    return "prose"


def _tail_by_tokens(text: str, tokenizer, min_tokens: int) -> str:
    """Extract the last `min_tokens` tokens' worth of words from `text`.

    HybridChunker has no native overlap between chunks, so this
    reconstructs overlap by taking the previous chunk's tail.
    """
    words = text.split()
    if not words or min_tokens <= 0:
        return ""
    for k in range(1, len(words) + 1):
        candidate = " ".join(words[-k:])
        if tokenizer.count_tokens(candidate) >= min_tokens:
            return candidate
    return " ".join(words)


print("All Docling helper functions defined:")
print("  _item_label, _resolve_doc_item, _try_table_markdown,")
print("  _try_table_structured, _extract_table_cells,")
print("  get_chunk_type, _tail_by_tokens")

All Docling helper functions defined:
  _item_label, _resolve_doc_item, _try_table_markdown,
  _try_table_structured, _extract_table_cells,
  get_chunk_type, _tail_by_tokens


### Define the Hybrid Chunker runner

This function builds a `HybridChunker` with configurable parameters, runs
it on a `DoclingDocument`, and returns both the chunk text strings (for our
diagnostic checks) and the full chunk records (for detailed inspection).

In [26]:
def run_hybrid_chunker(
    doc,
    max_tokens: int | None = None,
    repeat_table_header: bool = True,
    merge_peers: bool = True,
    always_emit_headings: bool = True,
    use_contextualize: bool = True,
    overlap_tokens: int = 0,
    min_chunk_words: int = MIN_CHUNK_WORDS,
) -> tuple[list[str], list[dict]]:
    """Run HybridChunker and return emitted text plus diagnostic records.

    This benchmark keeps the text emitted by HybridChunker as the canonical
    chunk text. A complete table export is retained separately for inspection;
    replacing every split piece with the complete table would invalidate the
    token-budget comparison.
    """
    kwargs = dict(
        repeat_table_header=repeat_table_header,
        merge_peers=merge_peers,
        always_emit_headings=always_emit_headings,
    )
    if max_tokens is not None:
        kwargs["max_tokens"] = max_tokens

    chunker = HybridChunker(**kwargs)
    raw_chunks = list(chunker.chunk(doc))

    chunk_texts = []
    chunk_records = []
    prev_text = ""
    kept = 0

    for idx, c in enumerate(raw_chunks):
        ctype = get_chunk_type(c, doc)
        raw_text = getattr(c, "text", "") or ""
        raw_token_count = chunker.tokenizer.count_tokens(raw_text)

        if use_contextualize:
            embed_text = chunker.contextualize(c) or raw_text
        else:
            embed_text = raw_text

        doc_items = list(getattr(getattr(c, "meta", None), "doc_items", None) or [])
        table_md = ""
        table_structured = None
        table_cells = None
        if ctype == "table":
            table_md = _try_table_markdown(doc_items, doc)
            table_structured = _try_table_structured(doc_items, doc)
            table_cells = _extract_table_cells(doc_items, doc)

        if ctype == "prose" and len(raw_text.split()) < min_chunk_words:
            prev_text = raw_text
            continue

        if overlap_tokens > 0 and prev_text and kept > 0:
            prefix = _tail_by_tokens(prev_text, chunker.tokenizer, overlap_tokens)
            if prefix:
                embed_text = f"{prefix}\n\n{embed_text}"

        headings = getattr(getattr(c, "meta", None), "headings", None) or []
        section_path = " > ".join(str(h).strip() for h in headings if str(h).strip())

        pages = set()
        for item in doc_items:
            for prov in getattr(item, "prov", []):
                pn = getattr(prov, "page_no", None)
                if pn is not None:
                    pages.add(pn)

        final_token_count = chunker.tokenizer.count_tokens(embed_text)
        record = {
            "chunk_id": f"fixture::chunk::{kept}",
            "chunk_index": kept,
            "source_chunk_index": idx,
            "chunk_type": ctype,
            "is_table": ctype == "table",
            "is_figure": ctype == "figure",
            "text": raw_text,
            "embed_text": embed_text,
            "raw_token_count": raw_token_count,
            "token_count": final_token_count,
            "budget": max_tokens,
            "budget_exceeded": bool(max_tokens is not None and final_token_count > max_tokens),
            "section_path": section_path,
            "pages": sorted(pages),
            "table_markdown": table_md,
            "table_structured": table_structured,
            "table_cells_count": len(table_cells) if table_cells else 0,
        }

        chunk_texts.append(raw_text)
        chunk_records.append(record)
        prev_text = raw_text
        kept += 1

    return chunk_texts, chunk_records


print("run_hybrid_chunker defined.")

run_hybrid_chunker defined.


### Run the Hybrid Chunker across parameter configurations

In [27]:
hybrid_results = []

hybrid_configs = [
    # (label, max_tokens, repeat_header, merge_peers, emit_headings, contextualize, overlap)
    ("hybrid-256tok",               256,  True,  True,  True,  True,  0),
    ("hybrid-512tok",               512,  True,  True,  True,  True,  0),
    ("hybrid-1024tok",             1024,  True,  True,  True,  True,  0),
    ("hybrid-512tok-no-hdr-repeat", 512,  False, True,  True,  True,  0),
    ("hybrid-512tok-no-merge",      512,  True,  False, True,  True,  0),
    ("hybrid-512tok-no-headings",   512,  True,  True,  False, True,  0),
    ("hybrid-512tok-no-context",    512,  True,  True,  True,  False, 0),
    ("hybrid-512tok-overlap50",     512,  True,  True,  True,  True,  50),
]

print("=" * 72)
print("STRATEGY 3: Docling Hybrid Chunker")
print("=" * 72)
print("This sweep evaluates chunk structure and token budgets only.")
print("No production embedding model is called here.")

# Store full records for the best config for later deep-inspection
best_records = None

for label, max_tok, repeat_hdr, merge, emit_hd, ctx, ov in hybrid_configs:
    print(f"\n--- {label} ---")
    texts, records = run_hybrid_chunker(
        docling_doc,
        max_tokens=max_tok,
        repeat_table_header=repeat_hdr,
        merge_peers=merge,
        always_emit_headings=emit_hd,
        use_contextualize=ctx,
        overlap_tokens=ov,
    )
    report = run_all_checks(texts, label)

    n_table = sum(1 for r in records if r["chunk_type"] == "table")
    n_prose = sum(1 for r in records if r["chunk_type"] == "prose")
    n_figure = sum(1 for r in records if r["chunk_type"] == "figure")
    n_over_budget = sum(1 for r in records if r["budget_exceeded"])
    print(f"  Chunk types: {n_prose} prose, {n_table} table, {n_figure} figure")
    print(f"  Final payloads over budget: {n_over_budget}/{len(records)}")

    n_ctx = sum(1 for r in records if r["embed_text"] != r["text"])
    print(f"  Contextualized chunks (embed_text != text): {n_ctx}/{len(records)}")

    report["n_table"] = n_table
    report["n_prose"] = n_prose
    report["n_figure"] = n_figure
    report["n_contextualized"] = n_ctx
    report["n_over_budget"] = n_over_budget
    hybrid_results.append(report)

    if label == "hybrid-512tok":
        best_records = records

print("\n" + "=" * 72)

STRATEGY 3: Docling Hybrid Chunker
This sweep evaluates chunk structure and token budgets only.
No production embedding model is called here.

--- hybrid-256tok ---


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (956 > 512). Running this sequence through the model will result in indexing errors


[hybrid-256tok] 20 chunks, 30–139 words, mean 77.2
  Table header retention : 100% (0/0 table chunks keep header)
  Code block integrity   : 3/5 intact
  Math delimiter balance : 4/4 balanced
  Q&A same chunk         : True
  Diagram+caption same   : False
  Chunk types: 16 prose, 4 table, 0 figure
  Final payloads over budget: 3/20
  Contextualized chunks (embed_text != text): 20/20

--- hybrid-512tok ---


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (956 > 512). Running this sequence through the model will result in indexing errors


[hybrid-512tok] 11 chunks, 32–247 words, mean 140.4
  Table header retention : 100% (0/0 table chunks keep header)
  Code block integrity   : 3/3 intact
  Math delimiter balance : 2/2 balanced
  Q&A same chunk         : True
  Diagram+caption same   : True
  Chunk types: 9 prose, 2 table, 0 figure
  Final payloads over budget: 1/11
  Contextualized chunks (embed_text != text): 11/11

--- hybrid-1024tok ---


...

[hybrid-512tok-no-headings] 11 chunks, 32–247 words, mean 140.4
  Table header retention : 100% (0/0 table chunks keep header)
  Code block integrity   : 3/3 intact
  Math delimiter balance : 2/2 balanced
  Q&A same chunk         : True
  Diagram+caption same   : True
  Chunk types: 9 prose, 2 table, 0 figure
  Final payloads over budget: 1/11
  Contextualized chunks (embed_text != text): 11/11

--- hybrid-512tok-no-context ---


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (956 > 512). Running this sequence through the model will result in indexing errors


[hybrid-512tok-no-context] 11 chunks, 32–247 words, mean 140.4
  Table header retention : 100% (0/0 table chunks keep header)
  Code block integrity   : 3/3 intact
  Math delimiter balance : 2/2 balanced
  Q&A same chunk         : True
  Diagram+caption same   : True
  Chunk types: 9 prose, 2 table, 0 figure
  Final payloads over budget: 0/11
  Contextualized chunks (embed_text != text): 0/11

--- hybrid-512tok-overlap50 ---


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (956 > 512). Running this sequence through the model will result in indexing errors


[hybrid-512tok-overlap50] 11 chunks, 32–247 words, mean 140.4
  Table header retention : 100% (0/0 table chunks keep header)
  Code block integrity   : 3/3 intact
  Math delimiter balance : 2/2 balanced
  Q&A same chunk         : True
  Diagram+caption same   : True
  Chunk types: 9 prose, 2 table, 0 figure
  Final payloads over budget: 2/11
  Contextualized chunks (embed_text != text): 11/11



### Deep inspection: the hybrid-512tok configuration

Let's look at every chunk from the default Hybrid config in detail —
including chunk_type, section_path, token count, and whether tables were
exported cleanly.

In [28]:
if best_records:
    print(f"{'idx':>3}  {'type':>6}  {'raw':>6}  {'final':>6}  {'budget':>6}  {'section_path'}")
    print("-" * 96)
    for r in best_records:
        sp = r['section_path'] if r['section_path'] else '(no heading)'
        budget = str(r['budget']) if r['budget'] is not None else '-'
        status = " OVER" if r["budget_exceeded"] else ""
        print(f"{r['chunk_index']:3d}  {r['chunk_type']:>6}  {r['raw_token_count']:6d}  "
              f"{r['token_count']:6d}  {budget:>6}  {sp}{status}")

    table_chunks = [r for r in best_records if r["chunk_type"] == "table"]
    if table_chunks:
        print("\n--- First table chunk emitted by HybridChunker ---")
        print(table_chunks[0]["text"][:500])
        print("\n--- Complete table export retained for inspection ---")
        print(table_chunks[0]["table_markdown"][:500] if table_chunks[0]["table_markdown"] else "(no markdown export)")

        if table_chunks[0]["table_structured"]:
            ts = table_chunks[0]["table_structured"]
            print(f"\n--- Structured export: {ts['num_rows']} rows x {ts['num_cols']} cols ---")
            print(f"Headers: {ts['headers']}")
else:
    print("No records found for hybrid-512tok.")

idx    type     raw   final  budget  section_path
------------------------------------------------------------------------------------------------
  0   prose      39      53     512  Ghana Primary Mathematics Curriculum — Upper Primary (B4–B6)
  1   prose     265     283     512  Ghana Primary Mathematics Curriculum — Upper Primary (B4–B6) > Rationale and Philosophy
  2   prose      62     111     512  Ghana Primary Mathematics Curriculum — Upper Primary (B4–B6) > Strand 1: Number > Sub-strand 1: Number and Numeration > B4.1.1.1 — Count, Read and Write Whole Numbers up to 100,000
  3   prose      65     109     512  Ghana Primary Mathematics Curriculum — Upper Primary (B4–B6) > Strand 1: Number > Sub-strand 2: Number Operations > B4.1.2.1 — Addition and Subtraction of Four-Digit Numbers
  4   prose      49      67     512  Ghana Primary Mathematics Curriculum — Upper Primary (B4–B6) > Content Standards and Indicators
  5   table     512     530     512  Ghana Primary Mathematics Curri

### Visual inspection of emitted chunks

The metrics tell us that a boundary or delimiter failed; this view shows the
actual text. It uses the `hybrid-512tok` records and displays one representative
record for each available chunk type. `text` is the emitted chunk, while
`embed_text` is the enriched text that would be passed to a later embedding
stage. This notebook does not call that embedding stage.

In [29]:
from IPython.display import HTML, display
from html import escape


def show_record(record: dict) -> None:
    budget = record["budget"] if record["budget"] is not None else "not set"
    status = "OVER BUDGET" if record["budget_exceeded"] else "within budget"
    metadata = (
        f"source={record['source_chunk_index']} | type={record['chunk_type']} | "
        f"section={record['section_path'] or '(no heading)'} | "
        f"raw_tokens={record['raw_token_count']} | final_tokens={record['token_count']} | "
        f"budget={budget} | {status}"
    )
    body = (
        f"<h4>{escape(metadata)}</h4>"
        f"<p><strong>Emitted chunk text</strong></p>"
        f"<pre>{escape(record['text'])}</pre>"
        f"<p><strong>Embedding text candidate</strong></p>"
        f"<pre>{escape(record['embed_text'])}</pre>"
    )
    display(HTML(body))


if best_records:
    seen_types = set()
    for record in best_records:
        if record["chunk_type"] in seen_types:
            continue
        show_record(record)
        seen_types.add(record["chunk_type"])
else:
    print("No records available for visual inspection.")

### Compare: repeat_table_header=True vs False

This is a critical Hybrid-only feature. When a large table exceeds the token
budget, the chunker must split it. With `repeat_table_header=True`, every
resulting chunk gets the column headers prepended. With `False`, only the
first chunk has headers — exactly like the Fixed splitter's failure mode.

In [32]:
# Run with a very small token budget to force table splitting.
# The emitted table text is Docling's chunk representation; the complete
# Markdown export is retained separately in each record for inspection.
print("--- Forcing table split with max_tokens=128 ---\n")

for repeat_hdr in [True, False]:
    label = f"128tok-repeat_header={repeat_hdr}"
    texts, records = run_hybrid_chunker(
        docling_doc, max_tokens=128,
        repeat_table_header=repeat_hdr,
    )
    table_recs = [r for r in records if r["chunk_type"] == "table"]
    print(f"  [{label}] {len(table_recs)} emitted table chunks")
    for tr in table_recs:
        preview = " ".join(tr["text"].split())[:140]
        print(
            f"    source={tr['source_chunk_index']}: "
            f"{tr['token_count']} final tokens, "
            f"budget exceeded={tr['budget_exceeded']}, "
            f"preview={preview!r}"
        )
    print("  Complete Markdown export is shown in the visual inspection section.")
    print()

--- Forcing table split with max_tokens=128 ---



[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (956 > 512). Running this sequence through the model will result in indexing errors


  [128tok-repeat_header=True] 8 emitted table chunks
    source=7: 141 final tokens, budget exceeded=True, preview='B4.1.1.1 Round whole numbers, Indicator = B4.1.1.1.5 Round a number to the nearest ten, hundred or thousand. B4.1.1.1 Round whole numbers, E'
    source=8: 145 final tokens, budget exceeded=True, preview='B4.1.1.2 Compare and order, Indicator = B4.1.1.2.1 Use inequality symbols to compare numbers up to 100,000. B4.1.1.2 Compare and order, Exem'
    source=9: 146 final tokens, budget exceeded=True, preview='Indicator = B5.1.2.1.3 Multiply a 3-digit number by a 2-digit number using the standard algorithm. B5.1.2.1 Multiply multi-digit numbers, Ex'
    source=10: 140 final tokens, budget exceeded=True, preview='B5.1.3.1 Fractions and decimals, Indicator = B5.1.3.1.2 Convert between fractions and decimals. B5.1.3.1 Fractions and decimals, Exemplar = '
    source=11: 144 final tokens, budget exceeded=True, preview='B6.1.2.1 Order of operations, Indicator = B6.1.2.1.1 Apply BOD

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (956 > 512). Running this sequence through the model will result in indexing errors


  [128tok-repeat_header=False] 10 emitted table chunks
    source=7: 116 final tokens, budget exceeded=False, preview='B4.1.1.1 Round whole numbers, Indicator = B4.1.1.1.5 Round a number to the nearest ten, hundred or thousand. B4.1.1.1 Round whole numbers, E'
    source=8: 114 final tokens, budget exceeded=False, preview='B4.1.1.1 Round whole numbers, ICT Integration = Use a spreadsheet ROUND function to verify. B4.1.1.2 Compare and order, Indicator = B4.1.1.2'
    source=9: 104 final tokens, budget exceeded=False, preview='B4.1.1.2 Compare and order, Level = B4. B4.1.1.2 Compare and order, ICT Integration = Display numbers on a number line applet. B5.1.2.1 Mult'
    source=10: 116 final tokens, budget exceeded=False, preview='B5.1.2.1 Multiply multi-digit numbers, Exemplar = 234 × 56 = 13,104 (partial products: 234×6=1,404; 234×50=11,700). B5.1.2.1 Multiply multi-'
    source=11: 114 final tokens, budget exceeded=False, preview='B5.1.3.1 Fractions and decimals, Indicator = B5.1.3.1.2

---

## Step 7 — Comparative Benchmark Table

Now we assemble all results into a single comparison table. Each row is one
configuration; columns show the diagnostic metrics across all 7 data types.

In [31]:
all_results = fixed_results + semantic_results + hybrid_results

# Build the comparison table
header = (f"{'Configuration':<30} {'Chunks':>6} {'Words':>12} "
          f"{'Table%':>7} {'Code':>5} {'Math':>5} {'Q&A':>5} {'Diag':>5}")

print("=" * len(header))
print("COMPARATIVE BENCHMARK TABLE")
print("=" * len(header))
print(header)
print("-" * len(header))

for r in all_results:
    word_range = f"{r['min_words']}–{r['max_words']}"
    table_pct = f"{r['table_retention']:.0%}"
    code_ok   = "✓" if r['code_intact'] else "✗"
    math_ok   = "✓" if r['math_balanced'] else "✗"
    qa_ok     = "✓" if r['qa_together'] else "✗"
    diag_ok   = "✓" if r['diagram_together'] else "✗"

    print(f"{r['label']:<30} {r['count']:>6} {word_range:>12} "
          f"{table_pct:>7} {code_ok:>5} {math_ok:>5} {qa_ok:>5} {diag_ok:>5}")

print("-" * len(header))
print()
print("Legend:")
print("  Table%  = % of table-data chunks that retained the column header row")
print("  Code    = ✓ if all code fence markers (```) are balanced in every chunk")
print("  Math    = ✓ if all $ delimiters are balanced in every chunk")
print("  Q&A     = ✓ if question prompt and student answer are in the same chunk")
print("  Diag    = ✓ if diagram/chart and its caption are in the same chunk")

COMPARATIVE BENCHMARK TABLE
Configuration                  Chunks        Words  Table%  Code  Math   Q&A  Diag
----------------------------------------------------------------------------------
default-200w-0ov                   10       58–195     50%     ✓     ✓     ✓     ✓
default-400w-0ov                    4      352–399    100%     ✓     ✓     ✗     ✓
default-400w-50ov                   5       97–392    100%     ✓     ✓     ✓     ✓
default-800w-0ov                    2      734–789    100%     ✗     ✓     ✓     ✓
default-800w-50ov                   2      784–789    100%     ✗     ✓     ✓     ✓
heading-400w-50ov                   5      190–381    100%     ✓     ✓     ✓     ✓
heading-800w-50ov                   2      735–788    100%     ✓     ✓     ✓     ✓
markdown-400w-50ov                  6       57–395    100%     ✗     ✓     ✓     ✓
markdown-800w-50ov                  3      190–789    100%     ✗     ✓     ✓     ✓
semantic-τ=0.50                     1    1523–1523    100% 

---

## Step 8 — Final Takeaways & Synthesis

### What we learned

**1. Fixed / Recursive splitting** is fast and simple, but it has no
structural awareness:
- **Tables**: The splitter cuts directly through table rows when the word
  budget is reached. The second chunk has data values with no column header,
  making them uninterpretable. Neither the default separators, the
  heading-aware separators, nor the Markdown-aware `MarkdownTextSplitter`
  fix this — none of them have a concept of "table row" or "table header".
- **Code blocks**: `MarkdownTextSplitter` helps by splitting on code fence
  boundaries, but the default and heading-aware separators can slice through
  code mid-function.
- **Larger `chunk_size` helps tables** (800w fits more rows) but **hurts
  retrieval precision** (chunks become diluted with irrelevant text).

**2. Semantic / Adaptive chunking** groups text by meaning shifts, but it
has critical blind spots:
- **Tables**: Pipe-delimited rows lack `.!?` punctuation, so the sentence
  splitter either treats the whole table as one huge "sentence" or breaks on
  decimal points inside numbers (e.g. `3.14` → `3` and `14`).
- **Code**: Code lines end with `:`, `)`, or nothing — not `.!?`. The
  sentence splitter cannot find meaningful boundaries.
- **Threshold sensitivity**: High τ (0.90) over-fragments into tiny chunks;
  low τ (0.50) merges unrelated sections together.
- **Offline embedding caveat**: Our `hash_embed_stub` is NOT semantically
  meaningful — with a real embedding model, the grouping would be different.
  But the sentence-splitting problem persists regardless of embedding quality.

**3. Docling Hybrid Chunker** is the only strategy that handles all 7 data
types robustly:
- **Tables** are preserved as `TableItem` objects — they stay as single
  chunks when they fit the token budget, and `repeat_table_header=True`
  repeats column headers when a split is forced.
- **Code blocks** are kept intact within their structural boundaries.
- **Headings** are attached to every chunk via `section_path`.
- **`contextualize()`** enriches embedding text with hierarchical context.
- **`export_to_markdown()`** produces clean table representations.
- **`chunk_type` tagging** labels chunks as `"table"` / `"figure"` / `"prose"`.
- **Trade-off**: Requires converting text to a `DoclingDocument` first
  (adds ~1-2 seconds per document for markdown, more for PDF with layout
  analysis).

### Recommended configuration for downstream retrieval

| Document type | Recommended chunker | Key settings |
| --- | --- | --- |
| **Documents with tables, code, or math** | Docling Hybrid | `max_tokens=512`, `repeat_table_header=True`, `merge_peers=True`, `always_emit_headings=True`, `contextualize()` on |
| **Plain prose only (no structure)** | Fixed (MarkdownTextSplitter) or Semantic | `chunk_size=400w`, `overlap=50w` |
| **Mixed / unknown content** | Docling Hybrid | Safe default — it degrades gracefully to prose-like chunking when there's no structure to exploit |